# LLM Results Analysis

In [1]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

In [2]:
main_pattern = re.compile(r'^(?P<dataset>.+)_(?P<method>baseline|baselineNoPrototype|noPrototype|rulebased|jumbled)_(?P<k>\d+)_(?P<num_rules>\d+)_llm_results\.jsonl$')

candidate_bases = {
    Path('.'),
    #Path('llm_results'),
    #Path('../llm_results'),
    #Path('results/llm_results'),
    #Path('../results/llm_results'),
    #Path.home() / 'Documents' / 'GitHub' / 'TSC_LLM_rules' / 'results' / 'llm_results',
}

for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate_bases.add(parent / 'results' / 'llm_results')
    candidate_bases.add(parent / 'llm_results')

def count_main_named_files(candidate):
    if not candidate.exists() or not candidate.is_dir():
        return -1
    count = 0
    for path in candidate.rglob('*.jsonl'):
        if path.parent.name == 'old':
            continue
        if main_pattern.match(path.name):
            count += 1
    return count

scored_bases = []
for candidate in sorted(candidate_bases):
    score = count_main_named_files(candidate)
    if score >= 0:
        scored_bases.append((score, 'Trash' not in str(candidate), candidate))

if not scored_bases:
    raise ValueError('Could not find any candidate llm_results directory.')

score, _, base = max(scored_bases)
if score == 0:
    raise ValueError('Found candidate directories, but none contain files matching dataset_method_k_numrules_llm_results.jsonl.')

rows = []

def build_file_info(path, row):
    row_method = row.get('mode', '')

    canonical_name = f"{row['dataset']}_{row_method}_{int(row['k'])}_{int(row['num_rules'])}_llm_results.jsonl"
    match = main_pattern.match(path.name)

    if match:
        file_dataset = match.group('dataset')
        file_method = match.group('method')
        file_k = int(match.group('k'))
        file_num_rules = int(match.group('num_rules'))
    else:
        file_dataset = row['dataset']
        file_method = row_method
        file_k = int(row['k'])
        file_num_rules = int(row['num_rules'])

    is_main_file = path.name == canonical_name
    variant = 'main' if is_main_file else path.stem.replace('_llm_results', '')

    return {
        'dataset_from_filename': file_dataset,
        'mode_from_filename': file_method,
        'k_from_filename': file_k,
        'num_rules_from_filename': file_num_rules,
        'variant': variant,
        'is_main_file': is_main_file,
    }

for path in sorted(base.rglob('*.jsonl')):
    if path.name == 'old_llm_rules_results.jsonl' or path.parent.name == 'old':
        continue

    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        info = build_file_info(path, row)
        row['source_file'] = path.name
        row['source_relpath'] = path.relative_to(base).as_posix()
        row['source_bucket'] = path.parent.name
        row.update(info)
        if row['is_main_file']:
            row['dataset'] = info['dataset_from_filename']
            row['mode'] = info['mode_from_filename']
            row['k'] = info['k_from_filename']
            row['num_rules'] = info['num_rules_from_filename']
        rows.append(row)

df = pd.DataFrame(rows)
df = df.sort_values(['dataset', 'mode', 'k', 'num_rules', 'source_relpath']).reset_index(drop=True)

analysis_df = df[df['is_main_file']].copy().reset_index(drop=True)

print(f'Using results base: {base.resolve()}')
print(f'Matching main-pattern files in base: {score}')
print(f'Total runs loaded: {len(df)}')
print(f'Runs used in analysis: {len(analysis_df)}')
print(f'Main files found: {df[["source_relpath", "is_main_file"]].drop_duplicates()["is_main_file"].sum()}')
print('\nFiles found:')
display(
    df[['source_relpath', 'dataset_from_filename', 'mode_from_filename', 'k_from_filename', 'num_rules_from_filename', 'variant', 'is_main_file']]
    .drop_duplicates()
    .sort_values('source_relpath')
    .reset_index(drop=True)
)

display(analysis_df.head())

if analysis_df.empty:
    raise ValueError('analysis_df is empty. Check the base directory above and the listed files/flags.')

Using results base: C:\Users\chriz\Documents\UIB\Master Informatikk\Semester 3\TSC\LLM_rules_for_TSC\TSC_LLM_rules\results\llm_results
Matching main-pattern files in base: 35
Total runs loaded: 106
Runs used in analysis: 100
Main files found: 35

Files found:


,source_relpath,dataset_from_filename,mode_from_filename,k_from_filename,num_rules_from_filename,variant,is_main_file
0,Chinatown_baselineNoPrototype_2_2_llm_results....,Chinatown,baselineNoPrototype,2,2,main,True
1,Chinatown_baselineNoPrototype_3_2_llm_results....,Chinatown,baselineNoPrototype,3,2,main,True
2,Chinatown_baseline_2_2_llm_results.jsonl,Chinatown,baseline,2,2,main,True
3,Chinatown_baseline_3_2_llm_results.jsonl,Chinatown,baseline,3,2,main,True
4,Chinatown_noPrototype_3_3_llm_results.jsonl,Chinatown,noPrototype,3,3,main,True
5,Chinatown_rulebased_2_2_llm_results.jsonl,Chinatown,rulebased,2,2,main,True
6,Chinatown_rulebased_3_11_llm_results_med_low.j...,Chinatown,rulebased,3,11,Chinatown_rulebased_3_11_med_low,False
7,Chinatown_rulebased_3_12_llm_results.jsonl,Chinatown,rulebased,3,12,main,True
8,Chinatown_rulebased_3_1_llm_results.jsonl,Chinatown,rulebased,3,1,main,True
9,Chinatown_rulebased_3_2_llm_results.jsonl,Chinatown,rulebased,3,2,main,True


,dataset,mode,classifier,llm,k,num_rules,accuracy,extracted_rules,instance,repetition,support_examples,source_file,source_relpath,source_bucket,dataset_from_filename,mode_from_filename,k_from_filename,num_rules_from_filename,variant,is_main_file
0,Chinatown,baseline,miniRocket,gpt-5.1,2,2,0.73,{},"[{'instance_id': 1, 'ts_idx': 239, 'true_label...",1,"[{'class_label': 0, 'selection_type': 'prototy...",Chinatown_baseline_2_2_llm_results.jsonl,Chinatown_baseline_2_2_llm_results.jsonl,llm_results,Chinatown,baseline,2,2,main,True
1,Chinatown,baseline,miniRocket,gpt-5.1,2,2,0.72,{},"[{'instance_id': 1, 'ts_idx': 129, 'true_label...",2,"[{'class_label': 0, 'selection_type': 'prototy...",Chinatown_baseline_2_2_llm_results.jsonl,Chinatown_baseline_2_2_llm_results.jsonl,llm_results,Chinatown,baseline,2,2,main,True
2,Chinatown,baseline,miniRocket,gpt-5.1,2,2,0.70,{},"[{'instance_id': 1, 'ts_idx': 236, 'true_label...",3,"[{'class_label': 0, 'selection_type': 'prototy...",Chinatown_baseline_2_2_llm_results.jsonl,Chinatown_baseline_2_2_llm_results.jsonl,llm_results,Chinatown,baseline,2,2,main,True
3,Chinatown,baseline,miniRocket,gpt-5.1,3,2,0.80,{},"[{'instance_id': 1, 'ts_idx': 159, 'true_label...",1,NaN,Chinatown_baseline_3_2_llm_results.jsonl,Chinatown_baseline_3_2_llm_results.jsonl,llm_results,Chinatown,baseline,3,2,main,True
4,Chinatown,baseline,miniRocket,gpt-5.1,3,2,0.65,{},"[{'instance_id': 1, 'ts_idx': 121, 'true_label...",2,NaN,Chinatown_baseline_3_2_llm_results.jsonl,Chinatown_baseline_3_2_llm_results.jsonl,llm_results,Chinatown,baseline,3,2,main,True


In [3]:
run_counts = (
    analysis_df.groupby(['dataset', 'mode'])
      .size()
      .reset_index(name='n_runs')
      .sort_values(['dataset', 'mode'])
)

display(run_counts)

,dataset,mode,n_runs
0,Chinatown,baseline,6
1,Chinatown,baselineNoPrototype,6
2,Chinatown,noPrototype,3
3,Chinatown,rulebased,14
4,ECG200,baseline,3
5,ECG200,baselineNoPrototype,3
6,ECG200,noPrototype,3
7,ECG200,rulebased,17
8,SonyAIBORobotSurface1,baseline,3
9,SonyAIBORobotSurface1,baselineNoPrototype,3


## Mean Accuracy Per Configuration

In [4]:
summary = (
    analysis_df.groupby(['dataset', 'mode', 'k', 'num_rules'])['accuracy']
      .agg(['mean', 'std', 'count', 'min', 'max'])
      .reset_index()
      .sort_values(['dataset', 'mode', 'k', 'num_rules'])
)

summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
summary['min'] = summary['min'].round(3)
summary['max'] = summary['max'].round(3)

display(summary)

,dataset,mode,k,num_rules,mean,std,count,min,max
0,Chinatown,baseline,2,2,0.717,0.015,3,0.70,0.73
1,Chinatown,baseline,3,2,0.703,0.084,3,0.65,0.80
2,Chinatown,baselineNoPrototype,2,2,0.677,0.118,3,0.54,0.75
3,Chinatown,baselineNoPrototype,3,2,0.743,0.006,3,0.74,0.75
4,Chinatown,noPrototype,3,3,0.963,0.038,3,0.92,0.99
5,Chinatown,rulebased,2,2,0.897,0.045,3,0.85,0.94
6,Chinatown,rulebased,3,1,0.925,0.007,2,0.92,0.93
7,Chinatown,rulebased,3,2,0.820,0.099,2,0.75,0.89
8,Chinatown,rulebased,3,3,0.955,0.035,2,0.93,0.98
9,Chinatown,rulebased,3,12,0.960,0.010,3,0.95,0.97


In [5]:
for dataset in sorted(summary['dataset'].unique()):
    print(dataset)
    display(summary[summary['dataset'] == dataset].reset_index(drop=True))

Chinatown


,dataset,mode,k,num_rules,mean,std,count,min,max
0,Chinatown,baseline,2,2,0.717,0.015,3,0.70,0.73
1,Chinatown,baseline,3,2,0.703,0.084,3,0.65,0.80
2,Chinatown,baselineNoPrototype,2,2,0.677,0.118,3,0.54,0.75
3,Chinatown,baselineNoPrototype,3,2,0.743,0.006,3,0.74,0.75
4,Chinatown,noPrototype,3,3,0.963,0.038,3,0.92,0.99
5,Chinatown,rulebased,2,2,0.897,0.045,3,0.85,0.94
6,Chinatown,rulebased,3,1,0.925,0.007,2,0.92,0.93
7,Chinatown,rulebased,3,2,0.820,0.099,2,0.75,0.89
8,Chinatown,rulebased,3,3,0.955,0.035,2,0.93,0.98
9,Chinatown,rulebased,3,12,0.960,0.010,3,0.95,0.97


ECG200


,dataset,mode,k,num_rules,mean,std,count,min,max
0,ECG200,baseline,3,2,0.703,0.187,3,0.49,0.84
1,ECG200,baselineNoPrototype,3,2,0.623,0.153,3,0.53,0.80
2,ECG200,noPrototype,3,3,0.630,0.078,3,0.58,0.72
3,ECG200,rulebased,3,1,0.457,0.231,3,0.24,0.70
4,ECG200,rulebased,3,2,0.640,0.017,3,0.63,0.66
5,ECG200,rulebased,3,3,0.680,0.060,3,0.62,0.74
6,ECG200,rulebased,3,12,0.707,0.076,3,0.62,0.76
7,ECG200,rulebased,4,10,0.550,0.368,2,0.29,0.81
8,ECG200,rulebased,4,11,0.647,0.110,3,0.54,0.76


SonyAIBORobotSurface1


,dataset,mode,k,num_rules,mean,std,count,min,max
0,SonyAIBORobotSurface1,baseline,3,2,0.583,0.015,3,0.57,0.60
1,SonyAIBORobotSurface1,baselineNoPrototype,3,2,0.533,0.055,3,0.48,0.59
2,SonyAIBORobotSurface1,noPrototype,3,3,0.640,0.115,4,0.48,0.75
3,SonyAIBORobotSurface1,rulebased,3,1,0.550,0.061,3,0.51,0.62
4,SonyAIBORobotSurface1,rulebased,3,2,0.573,0.050,3,0.52,0.62
5,SonyAIBORobotSurface1,rulebased,3,3,0.573,0.067,3,0.50,0.63
6,SonyAIBORobotSurface1,rulebased,3,12,0.643,0.065,3,0.58,0.71
7,SonyAIBORobotSurface1,rulebased,9,12,0.440,0.099,2,0.37,0.51


UMD


,dataset,mode,k,num_rules,mean,std,count,min,max
0,UMD,baseline,3,2,0.383,0.064,3,0.31,0.43
1,UMD,baselineNoPrototype,3,2,0.393,0.055,3,0.34,0.45
2,UMD,noPrototype,3,3,0.740,0.192,3,0.52,0.87
3,UMD,rulebased,3,1,0.953,0.023,3,0.94,0.98
4,UMD,rulebased,3,2,0.910,0.052,3,0.85,0.94
5,UMD,rulebased,3,3,0.923,0.047,3,0.87,0.96
6,UMD,rulebased,3,12,0.863,0.101,3,0.77,0.97


In [6]:
for dataset in sorted(summary['dataset'].unique()):
    print('=' * 80)
    print(dataset)

    dataset_summary = summary[summary['dataset'] == dataset]
    best_rule = dataset_summary[dataset_summary['mode'] == 'rulebased'].sort_values('mean', ascending=False).iloc[0]
    print(f"Best rulebased config: k={int(best_rule['k'])}, r={int(best_rule['num_rules'])}, mean={best_rule['mean']:.3f}, std={best_rule['std']:.3f}")

    baseline_rows = dataset_summary[dataset_summary['mode'] == 'baseline'].sort_values('k')
    for _, row in baseline_rows.iterrows():
        print(f"Baseline k={int(row['k'])}: mean={row['mean']:.3f}")

    baseline_no_proto_rows = dataset_summary[dataset_summary['mode'] == 'baselineNoPrototype'].sort_values('k')
    for _, row in baseline_no_proto_rows.iterrows():
        print(f"BaselineNoPrototype k={int(row['k'])}: mean={row['mean']:.3f}")

    no_proto_rows = dataset_summary[dataset_summary['mode'] == 'noPrototype'].sort_values(['k', 'num_rules'])
    for _, row in no_proto_rows.iterrows():
        print(f"noPrototype k={int(row['k'])}, r={int(row['num_rules'])}: mean={row['mean']:.3f}")

    if not baseline_rows.empty:
        best_baseline = baseline_rows.sort_values('mean', ascending=False).iloc[0]
        print(f"Delta best_rulebased - best_baseline: {(best_rule['mean'] - best_baseline['mean']):+.3f}")

    if not no_proto_rows.empty:
        best_no_proto = no_proto_rows.sort_values('mean', ascending=False).iloc[0]
        print(f"Delta best_rulebased - best_noPrototype: {(best_rule['mean'] - best_no_proto['mean']):+.3f}")


Chinatown
Best rulebased config: k=3, r=12, mean=0.960, std=0.010
Baseline k=2: mean=0.717
Baseline k=3: mean=0.703
BaselineNoPrototype k=2: mean=0.677
BaselineNoPrototype k=3: mean=0.743
noPrototype k=3, r=3: mean=0.963
Delta best_rulebased - best_baseline: +0.243
Delta best_rulebased - best_noPrototype: -0.003
ECG200
Best rulebased config: k=3, r=12, mean=0.707, std=0.076
Baseline k=3: mean=0.703
BaselineNoPrototype k=3: mean=0.623
noPrototype k=3, r=3: mean=0.630
Delta best_rulebased - best_baseline: +0.004
Delta best_rulebased - best_noPrototype: +0.077
SonyAIBORobotSurface1
Best rulebased config: k=3, r=12, mean=0.643, std=0.065
Baseline k=3: mean=0.583
BaselineNoPrototype k=3: mean=0.533
noPrototype k=3, r=3: mean=0.640
Delta best_rulebased - best_baseline: +0.060
Delta best_rulebased - best_noPrototype: +0.003
UMD
Best rulebased config: k=3, r=1, mean=0.953, std=0.023
Baseline k=3: mean=0.383
BaselineNoPrototype k=3: mean=0.393
noPrototype k=3, r=3: mean=0.740
Delta best_rulebas

## LaTeX Tables By k

In [7]:
dataset_order = ['Chinatown', 'UMD', 'ECG200', 'SonyAIBORobotSurface1']
dataset_labels = {
    'Chinatown': 'Chinatown',
    'UMD': 'UMD',
    'ECG200': 'ECG200',
    'SonyAIBORobotSurface1': 'Sony..1',
}

table_summary = summary[summary['mode'].isin(['baseline', 'baselineNoPrototype', 'noPrototype', 'rulebased'])].copy()
table_summary['config'] = table_summary.apply(
    lambda row: 'baseline'
    if row['mode'] == 'baseline'
    else (
        'baseline random'
        if row['mode'] == 'baselineNoPrototype'
        else (
            f'random (r={int(row["num_rules"])})'
            if row['mode'] == 'noPrototype'
            else f'r={int(row["num_rules"])}'
        )
    ),
    axis=1
)

all_k_values = sorted(table_summary['k'].unique())

for k_value in all_k_values:
    subset = table_summary[table_summary['k'] == k_value].copy()

    row_order = ['baseline']
    if 'baseline random' in subset['config'].values:
        row_order += ['baseline random']
    row_order += sorted(subset.loc[subset['mode'] == 'noPrototype', 'config'].unique(), key=lambda x: int(x.split('=')[-1].rstrip(')')))
    row_order += sorted(subset.loc[subset['mode'] == 'rulebased', 'config'].unique(), key=lambda x: int(x.split('=')[-1]))

    mean_df = (
        subset.pivot_table(index='config', columns='dataset', values='mean', aggfunc='first')
        .reindex(index=row_order)
        .reindex(columns=dataset_order)
        .rename(columns=dataset_labels)
    )
    std_df = (
        subset.pivot_table(index='config', columns='dataset', values='std', aggfunc='first')
        .reindex(index=row_order)
        .reindex(columns=dataset_order)
        .rename(columns=dataset_labels)
    )

    table_df = mean_df.copy().astype(object)
    for row_name in table_df.index:
        for col_name in table_df.columns:
            mean_val = mean_df.loc[row_name, col_name]
            std_val = std_df.loc[row_name, col_name]
            if pd.notna(mean_val):
                table_df.loc[row_name, col_name] = f'{mean_val:.0%} ± {std_val:.0%}' if pd.notna(std_val) else f'{mean_val:.0%}'
            else:
                table_df.loc[row_name, col_name] = '-'

    table_df.index.name = 'Config'

    latex_table = table_df.to_latex(
    escape=False,
    column_format='l' + 'c' * len(table_df.columns),
    caption=f'Mean accuracy ± std for k={int(k_value)}',
    label=f'tab:mean_accuracy_k_{int(k_value)}'
    )

    print(f'Mean accuracy ± std for k={int(k_value)}')
    display(table_df)
    print()

    print(latex_table)



Mean accuracy ± std for k=2


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,72% ± 2%,-,-,-
baseline random,68% ± 12%,-,-,-
r=2,90% ± 4%,-,-,-



\begin{table}
\caption{Mean accuracy ± std for k=2}
\label{tab:mean_accuracy_k_2}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & 72% ± 2% & - & - & - \\
baseline random & 68% ± 12% & - & - & - \\
r=2 & 90% ± 4% & - & - & - \\
\bottomrule
\end{tabular}
\end{table}

Mean accuracy ± std for k=3


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,70% ± 8%,38% ± 6%,70% ± 19%,58% ± 2%
baseline random,74% ± 1%,39% ± 6%,62% ± 15%,53% ± 6%
random (r=3),96% ± 4%,74% ± 19%,63% ± 8%,64% ± 12%
r=1,92% ± 1%,95% ± 2%,46% ± 23%,55% ± 6%
r=2,82% ± 10%,91% ± 5%,64% ± 2%,57% ± 5%
r=3,96% ± 4%,92% ± 5%,68% ± 6%,57% ± 7%
r=12,96% ± 1%,86% ± 10%,71% ± 8%,64% ± 6%



\begin{table}
\caption{Mean accuracy ± std for k=3}
\label{tab:mean_accuracy_k_3}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & 70% ± 8% & 38% ± 6% & 70% ± 19% & 58% ± 2% \\
baseline random & 74% ± 1% & 39% ± 6% & 62% ± 15% & 53% ± 6% \\
random (r=3) & 96% ± 4% & 74% ± 19% & 63% ± 8% & 64% ± 12% \\
r=1 & 92% ± 1% & 95% ± 2% & 46% ± 23% & 55% ± 6% \\
r=2 & 82% ± 10% & 91% ± 5% & 64% ± 2% & 57% ± 5% \\
r=3 & 96% ± 4% & 92% ± 5% & 68% ± 6% & 57% ± 7% \\
r=12 & 96% ± 1% & 86% ± 10% & 71% ± 8% & 64% ± 6% \\
\bottomrule
\end{tabular}
\end{table}

Mean accuracy ± std for k=4


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,-,-,-,-
r=1,84% ± 8%,-,-,-
r=10,-,-,55% ± 37%,-
r=11,-,-,65% ± 11%,-



\begin{table}
\caption{Mean accuracy ± std for k=4}
\label{tab:mean_accuracy_k_4}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & - & - & - & - \\
r=1 & 84% ± 8% & - & - & - \\
r=10 & - & - & 55% ± 37% & - \\
r=11 & - & - & 65% ± 11% & - \\
\bottomrule
\end{tabular}
\end{table}

Mean accuracy ± std for k=9


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,-,-,-,-
r=12,-,-,-,44% ± 10%



\begin{table}
\caption{Mean accuracy ± std for k=9}
\label{tab:mean_accuracy_k_9}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & - & - & - & - \\
r=12 & - & - & - & 44% ± 10% \\
\bottomrule
\end{tabular}
\end{table}

